In [0]:
df = spark.table("workspace.default.credit_risk_dataset")

In [0]:
df.show(5)

+-----------+---+--------------+---------------+--------------------+-----------+-------------+-------------+--------------------+------------+------+-----------+------------+
|customer_id|age|monthly_income|employment_type|years_in_current_job|loan_amount|interest_rate|tenure_months|existing_loans_count|credit_score|region|dpd_last_6m|default_flag|
+-----------+---+--------------+---------------+--------------------+-----------+-------------+-------------+--------------------+------------+------+-----------+------------+
|          1| 45|         19093|       Business|                13.5|    1158834|        18.36|           60|                   1|         664|  West|         58|           1|
|          2| 55|         48019|     Unemployed|                 1.4|      73341|        12.66|           60|                   5|         593|  East|         13|           1|
|          3| 29|         15395|       Business|                16.7|     938362|        19.08|           12|           

Check default rate

In [0]:
df.groupBy("default_flag").count().show()

+------------+-----+
|default_flag|count|
+------------+-----+
|           1|  566|
|           0|  434|
+------------+-----+



Describe

In [0]:
df.describe().show()

+-------+-----------------+------------------+-----------------+---------------+--------------------+-----------------+-----------------+------------------+--------------------+-----------------+------+-----------------+------------------+
|summary|      customer_id|               age|   monthly_income|employment_type|years_in_current_job|      loan_amount|    interest_rate|     tenure_months|existing_loans_count|     credit_score|region|      dpd_last_6m|      default_flag|
+-------+-----------------+------------------+-----------------+---------------+--------------------+-----------------+-----------------+------------------+--------------------+-----------------+------+-----------------+------------------+
|  count|             1000|              1000|             1000|           1000|                1000|             1000|             1000|              1000|                1000|             1000|  1000|             1000|              1000|
|   mean|            500.5|            4

Create feature loan to income ratio

In [0]:
from pyspark.sql.functions import col
df = df.withColumn(
  "Loan_to_income_ratio",
  col("loan_amount")/(col("monthly_income")*12)
)

df.select("loan_amount","monthly_income","Loan_to_income_ratio")

DataFrame[loan_amount: bigint, monthly_income: bigint, Loan_to_income_ratio: double]

In [0]:
    df.show(5)

+-----------+---+--------------+---------------+--------------------+-----------+-------------+-------------+--------------------+------------+------+-----------+------------+--------------------+
|customer_id|age|monthly_income|employment_type|years_in_current_job|loan_amount|interest_rate|tenure_months|existing_loans_count|credit_score|region|dpd_last_6m|default_flag|Loan_to_income_ratio|
+-----------+---+--------------+---------------+--------------------+-----------+-------------+-------------+--------------------+------------+------+-----------+------------+--------------------+
|          1| 45|         19093|       Business|                13.5|    1158834|        18.36|           60|                   1|         664|  West|         58|           1|   5.057848426124758|
|          2| 55|         48019|     Unemployed|                 1.4|      73341|        12.66|           60|                   5|         593|  East|         13|           1| 0.12727774422624377|
|          3| 2

Create DPD risk flag

In [0]:
from pyspark.sql.functions import when
df = df.withColumn(
  "high_dpd_flag",
  when(col("dpd_last_6m")>30,1).otherwise(0)
)

df.select("dpd_last_6m","high_dpd_flag").show(10)

+-----------+-------------+
|dpd_last_6m|high_dpd_flag|
+-----------+-------------+
|         58|            1|
|         13|            0|
|         68|            1|
|         45|            1|
|         71|            1|
|         67|            1|
|         38|            1|
|         40|            1|
|         44|            1|
|         69|            1|
+-----------+-------------+
only showing top 10 rows


Create credit score bucket

In [0]:
df = df.withColumn(
    "credit_score_bucket",
    when(col("credit_score") >= 750, "Excellent")
    .when(col("credit_score") >= 700, "Good")
    .when(col("credit_score") >= 650,"Average")
    .otherwise("Poor")
)

df.select("credit_score","credit_score_bucket").show(10)

+------------+-------------------+
|credit_score|credit_score_bucket|
+------------+-------------------+
|         664|            Average|
|         593|               Poor|
|         822|          Excellent|
|         836|          Excellent|
|         813|          Excellent|
|         697|            Average|
|         551|               Poor|
|         807|          Excellent|
|         805|          Excellent|
|         716|               Good|
+------------+-------------------+
only showing top 10 rows


Validate business logic

In [0]:
df.groupBy("high_dpd_flag","default_flag").count().show()

+-------------+------------+-----+
|high_dpd_flag|default_flag|count|
+-------------+------------+-----+
|            1|           1|  414|
|            0|           1|  152|
|            1|           0|  255|
|            0|           0|  179|
+-------------+------------+-----+



In [0]:
df.groupBy("credit_score_bucket","default_flag").count().show()

+-------------------+------------+-----+
|credit_score_bucket|default_flag|count|
+-------------------+------------+-----+
|            Average|           1|   91|
|               Poor|           1|  233|
|          Excellent|           1|  146|
|          Excellent|           0|  182|
|               Good|           0|   86|
|               Poor|           0|  111|
|               Good|           1|   96|
|            Average|           0|   55|
+-------------------+------------+-----+



Loan to income ratio

In [0]:
from pyspark.sql.functions import when,col

df = df.withColumn(
    "Lti_bucket",
    when(col("Loan_to_income_ratio") < 1, "Low")
    .when(col("Loan_to_income_ratio") < 2, "Moderate")
    .when(col("Loan_to_income_ratio") < 3, "High")
    .otherwise("Very High")

)

df.select("Loan_to_income_ratio","Lti_bucket").show(10)

+--------------------+----------+
|Loan_to_income_ratio|Lti_bucket|
+--------------------+----------+
|   5.057848426124758| Very High|
| 0.12727774422624377|       Low|
|   5.079365594890116| Very High|
|  0.5574485869409701|       Low|
|  2.0240549399354433|      High|
|   0.433706398969061|       Low|
|  0.9738758950926732|       Low|
|  0.9872992197782702|       Low|
|   1.028014112372398|  Moderate|
|   0.842773792517268|       Low|
+--------------------+----------+
only showing top 10 rows


In [0]:
df.groupBy("Lti_bucket").avg("default_flag").show()

+----------+-------------------+
|Lti_bucket|  avg(default_flag)|
+----------+-------------------+
| Very High|                1.0|
|       Low|0.36772046589018303|
|      High|  0.956989247311828|
|  Moderate| 0.7899159663865546|
+----------+-------------------+



In [0]:
df.groupBy("existing_loans_count").avg("default_flag").orderBy("existing_loans_count").show()

+--------------------+------------------+
|existing_loans_count| avg(default_flag)|
+--------------------+------------------+
|                   0|0.4421052631578947|
|                   1|0.5333333333333333|
|                   2|           0.53125|
|                   3|0.6032608695652174|
|                   4|0.6503496503496503|
|                   5| 0.653179190751445|
+--------------------+------------------+



Loan count to risk

In [0]:
df = df.withColumn(
    "loan_count_bucket",
    when(col("existing_loans_count") <= 1, "Low")
    .when(col("existing_loans_count") <= 3, "Moderate")
    .otherwise("High")
)

In [0]:
df.groupBy("loan_count_bucket").avg("default_flag").show()

+-----------------+------------------+
|loan_count_bucket| avg(default_flag)|
+-----------------+------------------+
|              Low|0.4823529411764706|
|             High|0.6518987341772152|
|         Moderate|0.5697674418604651|
+-----------------+------------------+



Loan count to income

In [0]:
df = df.withColumn(
  "multi_loan_high_lti_flag",
  when((col("existing_loans_count") >=3) & (col("Loan_to_income_ratio") > 2), 1)
  .otherwise(0)
)

In [0]:
df.groupBy("multi_loan_high_lti_flag").avg("default_flag").show()

+------------------------+------------------+
|multi_loan_high_lti_flag| avg(default_flag)|
+------------------------+------------------+
|                       0|0.5308108108108108|
|                       1|               1.0|
+------------------------+------------------+



Experience to loan 

In [0]:
df.groupby("years_in_current_job").avg("default_flag").orderBy("years_in_current_job").show(5)

+--------------------+------------------+
|years_in_current_job| avg(default_flag)|
+--------------------+------------------+
|                 0.5|               1.0|
|                 0.6|               0.5|
|                 0.7|              0.75|
|                 0.8|              0.25|
|                 0.9|0.8333333333333334|
+--------------------+------------------+
only showing top 5 rows


In [0]:
df = df.withColumn(
    "Job_stability_bucket",
    when(col("years_in_current_job") <2,"Low_stability")
    .when(col("years_in_current_job") <5, "Moderate_stability")
    .otherwise("High_stability")
)

In [0]:
df.groupBy("Job_stability_bucket").avg("default_flag").show()

+--------------------+------------------+
|Job_stability_bucket| avg(default_flag)|
+--------------------+------------------+
|      High_stability|0.5681528662420382|
|       Low_stability|0.5844155844155844|
|  Moderate_stability|0.5434782608695652|
+--------------------+------------------+



In [0]:
df = df.withColumn(
    "high_lti_low_stability_flag",
    when(
        (col("Loan_to_income_ratio") > 2) &
        (col("years_in_current_job") < 2),
        1).otherwise(0)
)

In [0]:
df.groupBy("high_lti_low_stability_flag").avg("default_flag").show()

+---------------------------+------------------+
|high_lti_low_stability_flag| avg(default_flag)|
+---------------------------+------------------+
|                          0|0.5593908629441624|
|                          1|               1.0|
+---------------------------+------------------+



Create EMI proxy

In [0]:
df.groupBy("tenure_months").avg("default_flag").orderBy("tenure_months").show()

+-------------+------------------+
|tenure_months| avg(default_flag)|
+-------------+------------------+
|           12|0.5515463917525774|
|           24|0.5714285714285714|
|           36|0.5135135135135135|
|           48|0.5870646766169154|
|           60|0.5982142857142857|
+-------------+------------------+



In [0]:
df = df.withColumn(
    "EMI_proxy",
    col("loan_amount")/col("tenure_months")
)

In [0]:
df.groupBy("default_flag").avg("EMI_proxy").show()

+------------+-----------------+
|default_flag|   avg(EMI_proxy)|
+------------+-----------------+
|           1|34052.59882214372|
|           0|22971.39425243216|
+------------+-----------------+



EMI to income ratio

In [0]:
df = df.withColumn(
  "EMI_to_income_ratio",
  (col("loan_amount")/col("tenure_months"))/col("monthly_income")
)

In [0]:
df.groupBy("default_flag").avg("EMI_to_income_ratio").show()

+------------+------------------------+
|default_flag|avg(EMI_to_income_ratio)|
+------------+------------------------+
|           1|      0.7081653641970013|
|           0|     0.25600556197171875|
+------------+------------------------+



In [0]:
df = df.withColumn(
    "emi_stress_bucket",
    when(col("EMI_to_income_ratio") <0.3, "Low")
    .when(col("EMI_to_income_ratio") <0.5, "Medium")
    .when(col("EMI_to_income_ratio") <0.6, "High")
    .otherwise("Very_High")
)

In [0]:
df.groupBy("emi_stress_bucket").avg("default_flag").orderBy("emi_stress_bucket").show()

+-----------------+-------------------+
|emi_stress_bucket|  avg(default_flag)|
+-----------------+-------------------+
|             High| 0.6857142857142857|
|              Low|0.38461538461538464|
|           Medium| 0.6648351648351648|
|        Very_High| 0.8403041825095057|
+-----------------+-------------------+



Export to CSV 

In [0]:
final_cols = [
    "age",
    "monthly_income",
    "loan_amount",
    "interest_rate",
    "tenure_months",
    "existing_loans_count",
    "credit_score",
    "dpd_last_6m",
    "years_in_current_job",
    "loan_to_income_ratio",
    "emi_to_income_ratio",
    "employment_type",
    "region",
    "default_flag"
]

df_final = df.select(final_cols)

df_final.printSchema()
df_final.show(5)

root
 |-- age: long (nullable = true)
 |-- monthly_income: long (nullable = true)
 |-- loan_amount: long (nullable = true)
 |-- interest_rate: decimal(22,2) (nullable = true)
 |-- tenure_months: long (nullable = true)
 |-- existing_loans_count: long (nullable = true)
 |-- credit_score: long (nullable = true)
 |-- dpd_last_6m: long (nullable = true)
 |-- years_in_current_job: decimal(21,1) (nullable = true)
 |-- loan_to_income_ratio: double (nullable = true)
 |-- emi_to_income_ratio: double (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- region: string (nullable = true)
 |-- default_flag: long (nullable = true)

+---+--------------+-----------+-------------+-------------+--------------------+------------+-----------+--------------------+--------------------+--------------------+---------------+------+------------+
|age|monthly_income|loan_amount|interest_rate|tenure_months|existing_loans_count|credit_score|dpd_last_6m|years_in_current_job|loan_to_income_ratio| emi_

In [0]:
pdf = df_final.toPandas()

In [0]:
pdf.corr(numeric_only=True)

,age,monthly_income,loan_amount,tenure_months,existing_loans_count,credit_score,dpd_last_6m,loan_to_income_ratio,emi_to_income_ratio,default_flag
age,1.000000,-0.015534,-0.016391,0.024043,0.009660,0.005688,0.000247,-0.023921,-0.030315,0.058466
monthly_income,-0.015534,1.000000,-0.024304,-0.011547,0.006273,-0.083714,0.090443,-0.644262,-0.455431,-0.349582
loan_amount,-0.016391,-0.024304,1.000000,0.047980,-0.007724,0.031306,0.039089,0.551532,0.374621,0.376115
tenure_months,0.024043,-0.011547,0.047980,1.000000,-0.008354,-0.007008,0.013734,0.025890,-0.427727,0.032590
existing_loans_count,0.009660,0.006273,-0.007724,-0.008354,1.000000,0.023407,-0.009436,-0.010036,0.023387,0.149041
credit_score,0.005688,-0.083714,0.031306,-0.007008,0.023407,1.000000,-0.016277,0.063414,0.019273,-0.200175
dpd_last_6m,0.000247,0.090443,0.039089,0.013734,-0.009436,-0.016277,1.000000,-0.036272,-0.016669,0.196099
loan_to_income_ratio,-0.023921,-0.644262,0.551532,0.025890,-0.010036,0.063414,-0.036272,1.000000,0.725803,0.464917
emi_to_income_ratio,-0.030315,-0.455431,0.374621,-0.427727,0.023387,0.019273,-0.016669,0.725803,1.000000,0.324095
default_flag,0.058466,-0.349582,0.376115,0.032590,0.149041,-0.200175,0.196099,0.464917,0.324095,1.000000


In [0]:
display(df_final)


age,monthly_income,loan_amount,interest_rate,tenure_months,existing_loans_count,credit_score,dpd_last_6m,years_in_current_job,loan_to_income_ratio,emi_to_income_ratio,employment_type,region,default_flag
45,19093,1158834,18.36,60,1,664,58,13.5,5.057848426124758,1.0115696852249516,Business,West,1
55,48019,73341,12.66,60,5,593,13,1.4,0.12727774422624377,0.025455548845248754,Unemployed,East,1
29,15395,938362,19.08,12,2,822,68,16.7,5.079365594890116,5.079365594890116,Business,West,1
44,132797,888330,9.24,24,4,836,45,12.8,0.5574485869409701,0.27872429347048505,Unemployed,West,0
34,60102,1459797,21.39,48,0,813,71,20.0,2.0240549399354433,0.5060137349838608,Self_Employed,East,1
52,92343,480597,10.95,12,2,697,67,8.0,0.433706398969061,0.433706398969061,Unemployed,South,1
40,60841,711019,20.07,36,1,551,38,6.4,0.9738758950926732,0.32462529836422443,Business,South,1
60,80789,957155,13.48,36,0,807,40,18.0,0.9872992197782702,0.3290997399260901,Unemployed,South,0
29,44311,546628,16.89,36,2,805,44,10.1,1.028014112372398,0.3426713707907994,Business,South,0
47,65101,658385,11.42,24,3,716,69,5.9,0.842773792517268,0.421386896258634,Unemployed,East,0
